# Chapter 8 - RAGAS, bootstrapped confidence intervals, and trajectory checks

The RAGAS cell needs `ragas` to import cleanly and an `OPENAI_API_KEY` (or another provider) to actually score anything; it degrades to a diagnostic message on either failure, without breaking the rest of the notebook. The bootstrapping and trajectory-check sections run fully offline.

## RAGAS today: a version-drift bug, and the fix

As of this writing, `pip install ragas` crashes on import: it still pulls in a `ChatVertexAI` path that `langchain-community` removed at the 1.0 boundary, and since that package was archived by its own maintainers in mid-2026, the fix is a pin (`langchain-community==0.3.30`) rather than a wait for an upstream release. This chapter's `requirements.txt` pins exactly that. This cell checks your environment live rather than assuming the pin is in place.

In [1]:
import langchain_community

print(f'langchain-community {langchain_community.__version__}')

try:
    import ragas
    print(f'ragas {ragas.__version__} imported cleanly')
    if langchain_community.__version__.startswith('0.3'):
        print('(pinned to the last pre-1.0-split release, on purpose - see above)')
    RAGAS_OK = True
except ModuleNotFoundError as e:
    print(f'ragas failed to import: {e}')
    print('Your environment does not match this chapter\'s requirements.txt; '
          "see the markdown cell above for the pin that fixes it.")
    RAGAS_OK = False

langchain-community 0.3.30


ragas 0.4.3 imported cleanly
(pinned to the last pre-1.0-split release, on purpose - see above)


## Scoring faithfulness against the Meridian corpus

`Faithfulness` breaks the answer into atomic factual statements and checks each one against the retrieved context individually, which is exactly the check that should catch the assistant citing the outdated 2019 baggage-fee page instead of the current one. The call shape below is verified against the installed `ragas` 0.4.3 API. It runs for real against a free, local judge, LFM2.5-1.2B-Instruct served through Ollama (`ollama pull LiquidAI/lfm2.5-1.2b-instruct`), and falls back to a hosted judge if `OPENAI_API_KEY` is set and Ollama isn't running.

In [2]:
import os
import sys
import urllib.request

sys.path.insert(0, os.path.abspath('../chapter4'))
from meridian import load_corpus

corpus = {doc['doc_id']: doc['text'] for doc in load_corpus()}


def ollama_running(url='http://localhost:11434/api/version', timeout=1):
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except OSError:
        return False


judge_client, judge_model = None, None
if RAGAS_OK and ollama_running():
    from openai import AsyncOpenAI
    judge_client = AsyncOpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
    judge_model = 'LiquidAI/lfm2.5-1.2b-instruct'
elif RAGAS_OK and os.environ.get('OPENAI_API_KEY'):
    from openai import AsyncOpenAI
    judge_client = AsyncOpenAI()
    judge_model = 'gpt-4o-mini'

if judge_client is not None:
    from ragas.llms import llm_factory
    from ragas.metrics.collections import Faithfulness

    faithfulness = Faithfulness(llm=llm_factory(judge_model, client=judge_client))
    result = await faithfulness.ascore(
        user_input='How much is a checked bag on a Meridian package fare?',
        response='A checked bag follows the 2019 fee schedule.',  # cites the outdated page
        retrieved_contexts=[corpus['baggage_current'], corpus['baggage_2019_outdated']],
    )
    print(f'judge: {judge_model}')
    print(result)
else:
    reason = 'ragas will not import' if not RAGAS_OK else 'no local Ollama and no OPENAI_API_KEY'
    print(f'Skipping a live RAGAS score because {reason}.')
    print('The chapter deliberately does not print an invented RAGAS number for this reason.')

judge: LiquidAI/lfm2.5-1.2b-instruct
MetricResult(value=1.0)


## RAGAS's algorithm, without the RAGAS package

`Faithfulness` decomposes the answer into atomic statements, checks each one against the context, and averages the verdicts; that's RAGAS's own mechanism, read directly out of its source, not a different technique standing in for it. What gets dropped below is the dependency, not the approach. Reimplemented against a plain `init_chat_model` object, no RAGAS import involved. Falls back to a canned, labeled score with no local Ollama and no `OPENAI_API_KEY`, same as the RAGAS cell above.

In [3]:
from langchain.chat_models import init_chat_model
from pydantic import BaseModel


class Statements(BaseModel):
    statements: list[str]


class Verdict(BaseModel):
    supported: bool


def check_faithfulness(judge, question: str, answer: str, context: str) -> float:
    """Decompose the answer into statements, verify each against the context."""
    statements = judge.with_structured_output(Statements).invoke(
        f'List the factual claims in this answer as short, standalone statements.\n\n'
        f'Question: {question}\nAnswer: {answer}'
    ).statements
    if not statements:
        return float('nan')
    verdicts = [
        judge.with_structured_output(Verdict).invoke(
            f'Context:\n{context}\n\nIs this statement fully supported by the '
            f'context above? Statement: {s}'
        ).supported
        for s in statements
    ]
    return sum(verdicts) / len(verdicts)


faithfulness_judge = None
if ollama_running():
    faithfulness_judge = init_chat_model(
        'LiquidAI/lfm2.5-1.2b-instruct', model_provider='ollama', temperature=0)
elif os.environ.get('OPENAI_API_KEY'):
    faithfulness_judge = init_chat_model('openai:gpt-4o-mini')

if faithfulness_judge is not None:
    score = check_faithfulness(
        faithfulness_judge,
        question='How much is a checked bag on a Meridian package fare?',
        answer='A checked bag follows the 2019 fee schedule.',
        context=corpus['baggage_current'],
    )
    print('faithfulness (no RAGAS):', score)
else:
    print('No local Ollama and no OPENAI_API_KEY; skipping the live check.')

faithfulness (no RAGAS): 1.0


## Bootstrapping a confidence interval over judge scores

RAGAS scores a metric per example and averages, with nothing built in to say how much that average would move under resampling (confirmed absent from its source and docs, not just undocumented).

In [4]:
import numpy as np


def bootstrap_ci(scores, n_resamples=10_000, ci=0.95, seed=0):
    """Resample the per-example scores with replacement to get a confidence interval."""
    rng = np.random.default_rng(seed)
    scores = np.asarray(scores, dtype=float)
    means = [rng.choice(scores, size=len(scores), replace=True).mean() for _ in range(n_resamples)]
    lo, hi = np.percentile(means, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return float(scores.mean()), float(lo), float(hi)

scores = [1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1]  # 17/20
mean, lo, hi = bootstrap_ci(scores)
print(f'pass rate: {mean:.2f}, 95% CI: [{lo:.2f}, {hi:.2f}]')

pass rate: 0.85, 95% CI: [0.70, 1.00]


17/20 looks solid until the interval says the true pass rate could plausibly be anywhere from 70% to 100%. Two systems scoring 17/20 and 15/20 on the same twenty-question set are not distinguishable at 95% confidence, however precisely each percentage is quoted.

## Prediction intervals from a judge's own confidence

Bootstrap resampling needs many scores to say anything. A second technique gets a comparable interval from a single judge call, provided the judge's API returns token log-probabilities alongside the rating: pull the probability the model assigned to every candidate score, accumulate the most probable ones until they cover a chosen confidence level, and report the resulting range.

In [5]:
import math


def score_interval(top_logprobs: dict, threshold: float = 0.9) -> tuple:
    """Turn a judge's token log-probabilities into a prediction interval."""
    probs = {score: math.exp(lp) for score, lp in top_logprobs.items()}
    total = sum(probs.values())
    ranked = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)
    covered, kept = 0.0, []
    for score, prob in ranked:
        kept.append(score)
        covered += prob / total
        if covered >= threshold:
            break
    return min(kept), max(kept), (min(kept) + max(kept)) / 2

print(score_interval({7: -0.1, 6: -3.2, 8: -3.5}))         # confident judge
print(score_interval({7: -0.8, 6: -1.4, 4: -1.6, 5: -2.3}))  # confused judge

(7, 7, 7.0)
(4, 7, 5.5)


## Meridian's itinerary graph

The node functions (`gather_node`, `draft_node`, `check_node`, `repair_node`, `review`, `export_node`) are chapter 4's own, defined in `chapter4/04_itinerary_assistant.ipynb`. What the trajectory checks below actually reason about is the graph's wiring, not what each node does internally, so this cell rebuilds the same topology with lightweight stand-in nodes and prints it to confirm it matches.

In [6]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class AssistantState(TypedDict, total=False):
    brief: dict
    evidence: dict
    plan: dict
    problems: list[str]
    attempts: int
    approved: bool


def route_after_validation(state) -> str:
    if not state['problems']:
        return 'review'
    return 'repair' if state.get('attempts', 0) < 2 else 'review'


def _stand_in(name):
    def node(state):
        return {}
    return node


gather_node, draft_node, check_node, repair_node, review, export_node = (
    _stand_in(n) for n in ['gather', 'draft', 'check', 'repair', 'review', 'export']
)

builder = StateGraph(AssistantState)
for name, node in [('gather', gather_node), ('draft', draft_node),
                   ('check', check_node), ('repair', repair_node),
                   ('review', review), ('export', export_node)]:
    builder.add_node(name, node)
builder.add_edge(START, 'gather')
builder.add_edge('gather', 'draft')
builder.add_edge('draft', 'check')
builder.add_conditional_edges('check', route_after_validation,
                              {'repair': 'repair', 'review': 'review'})
builder.add_edge('repair', 'check')
builder.add_edge('review', 'export')
builder.add_edge('export', END)
graph = builder.compile(checkpointer=InMemorySaver())

nodes = sorted(n for n in graph.get_graph().nodes if not n.startswith('__'))
print('nodes:', nodes)
for source, target in sorted((e.source, e.target) for e in graph.get_graph().edges):
    print(' ', source, '->', target)

nodes: ['check', 'draft', 'export', 'gather', 'repair', 'review']
  __start__ -> gather
  check -> repair
  check -> review
  draft -> check
  export -> __end__
  gather -> draft
  repair -> check
  review -> export


## Trajectory checks: efficiency and safety

The itinerary assistant from chapter 4 is a LangGraph pipeline, `gather -> draft -> check -> repair (looping back to check) -> review -> export`, with a human pause at `review`. Its own `route_after_validation` caps repair attempts at two before forcing the run to `review` regardless, so a correctly-running graph can't oscillate forever. The traces below are illustrative, shaped like that real node sequence: `clean` is what a capped, working run looks like, and `broken_cap` is what happens if `repair_node` ever failed to increment its own attempt counter.

In [7]:
def check_trajectory(steps, max_steps=8, max_cycle_repeats=3):
    """Flag a trajectory for excess length or a short cycle repeating itself."""
    problems = []
    if len(steps) > max_steps:
        problems.append(f'ran {len(steps)} steps, over the {max_steps}-step budget')
    for period in (1, 2, 3):
        window = period * max_cycle_repeats
        for start in range(len(steps) - window + 1):
            chunk = steps[start:start + window]
            if chunk == chunk[:period] * max_cycle_repeats:
                problems.append(
                    f'steps {start}-{start + window - 1} repeat '
                    f'{chunk[:period]} {max_cycle_repeats} times without progressing'
                )
                return problems
    return problems

clean = ['gather', 'draft', 'check', 'repair', 'check', 'review', 'export']
broken_cap = ['gather', 'draft'] + ['check', 'repair'] * 3 + ['check', 'review', 'export']

print('clean:', check_trajectory(clean))
print('broken_cap:', check_trajectory(broken_cap))

clean: []
broken_cap: ['ran 11 steps, over the 8-step budget', "steps 2-7 repeat ['check', 'repair'] 3 times without progressing"]


In [8]:
def check_policy(steps, gate='review', guarded='export'):
    """Flag a trajectory that reaches a guarded node without its gate first."""
    if guarded not in steps:
        return []
    if gate not in steps[:steps.index(guarded)]:
        return [f"'{guarded}' ran without '{gate}' first, the approval gate was bypassed"]
    return []

skipped_review = ['gather', 'draft', 'check', 'export']
print('skipped_review, check_trajectory:', check_trajectory(skipped_review))
print('skipped_review, check_policy:    ', check_policy(skipped_review))

skipped_review, check_trajectory: []
skipped_review, check_policy:     ["'export' ran without 'review' first, the approval gate was bypassed"]


`check_trajectory` reports nothing on `skipped_review`: it's short and has no repeating cycle. `check_policy` is what catches it. A length-and-repetition check and a policy-gate check answer different questions, and neither substitutes for the other.

## Two more checks from the Agent Error Taxonomy

Two of the six failure modes this chapter names but doesn't check close with pieces already in this notebook. Progress misassessment reuses chapter 4's `validate(plan, brief, matrix)` as a state-based evaluator: compare the agent's own done/stuck claim against what `validate()` actually found.

In [9]:
def check_progress_claim(agent_says_done: bool, problems: list) -> bool:
    """True if the agent's own done/stuck claim disagrees with validate()."""
    actually_done = not problems
    return agent_says_done != actually_done

print(check_progress_claim(True, []))
print(check_progress_claim(True, ['too long']))

False
True


A malformed tool call gets the same treatment chapter 4's structured extraction already uses: validate the arguments against a Pydantic model before the call goes out. `adjust_plan` is the same tool this notebook's trajectory trace record uses; `drop_stop` is supposed to be a city name, and a `3` there is exactly the malformed argument this check exists to catch.

In [10]:
from pydantic import BaseModel, ValidationError


class AdjustPlanArgs(BaseModel):
    drop_stop: str
    reason: str = ''


def check_tool_call_arguments(tool_call: dict, schema: type) -> list:
    """Flag a tool call whose arguments don't validate against its schema."""
    try:
        schema(**tool_call['arguments'])
        return []
    except ValidationError as e:
        return [err['msg'] for err in e.errors()]

good_call = {'tool': 'adjust_plan', 'arguments': {'drop_stop': 'Uppsala'}}
bad_call = {'tool': 'adjust_plan', 'arguments': {'drop_stop': 3}}
print(check_tool_call_arguments(good_call, AdjustPlanArgs))
print(check_tool_call_arguments(bad_call, AdjustPlanArgs))

[]
['Input should be a valid string']
